
# Bonus Exercise: Variational Auto-Encoder (VAE) in PyTorch

This bonus exercise extends the auto-encoder lab from deterministic reconstruction to **generative modeling**.

A regular auto-encoder learns

$$
x \rightarrow z \rightarrow \hat{x}.
$$

A variational auto-encoder learns a distribution over latent codes:

$$
q_\phi(z \mid x) = \mathcal{N}(\mu(x), \mathrm{diag}(\sigma^2(x))).
$$

Then it samples

$$
z = \mu + \sigma \odot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I),
$$

and decodes $z$ back into an image.

The loss has two terms:

$$
\mathcal{L}
=
\mathrm{reconstruction\ loss}
+
\beta \, D_{\mathrm{KL}}\left(q_\phi(z \mid x) \| p(z)\right),
$$

where $$p(z)=\mathcal{N}(0,I)$$

This notebook is designed to run on a laptop GPU with CUDA.


In [ ]:

# Bonus VAE setup
# This cell is self-contained. It does not require the previous lab cells.
# if no cude, install cuda-torch 
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = DEVICE == "cuda"

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


In [ ]:

# Data: Fashion-MNIST
# Use num_workers=0 for Windows/laptop compatibility.
# Increase num_workers to 2 or 4 if your machine handles it well.

BATCH_SIZE = 128

transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY
)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

xb, yb = next(iter(train_loader))
print("Batch shape:", xb.shape)

plt.figure(figsize=(8, 3))
grid = make_grid(xb[:16], nrow=8, padding=2)
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off")
plt.title("Fashion-MNIST examples")
plt.show()


In [ ]:

def show_images(images, title=None, nrow=8, figsize=(8, 3)):
    images = images.detach().cpu().clamp(0, 1)
    grid = make_grid(images, nrow=nrow, padding=2)
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    if title is not None:
        plt.title(title)
    plt.show()


def plot_history(history):
    plt.figure(figsize=(7, 4))
    plt.plot(history["loss"], label="total")
    plt.plot(history["recon"], label="reconstruction")
    plt.plot(history["kl"], label="KL")
    plt.xlabel("epoch")
    plt.ylabel("average loss per image")
    plt.title("VAE training curves")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Model

Complete the VAE model. The solution version has the full implementation.

In [ ]:

class ConvVAE(nn.Module):
    """
    Small convolutional VAE for 28 x 28 grayscale images.

    Encoder:
        image -> feature maps -> hidden vector -> mu, logvar

    Decoder:
        latent vector z -> feature maps -> image
    """
    def __init__(self, latent_dim=8):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),   # 28 -> 14
            nn.LeakyReLU(0.1),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 14 -> 7
            nn.LeakyReLU(0.1),
        )

        self.flatten_dim = 64 * 7 * 7

        self.fc_hidden = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, 256),
            nn.LeakyReLU(0.1),
        )

        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        self.fc_decode = nn.Sequential(
            nn.Linear(latent_dim, self.flatten_dim),
            nn.LeakyReLU(0.1),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                64, 32, kernel_size=3, stride=2, padding=1, output_padding=1
            ),  # 7 -> 14
            nn.LeakyReLU(0.1),

            nn.ConvTranspose2d(
                32, 1, kernel_size=3, stride=2, padding=1, output_padding=1
            ),  # 14 -> 28
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = self.fc_hidden(h)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        if not self.training:
            return mu
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), 64, 7, 7)
        x_hat = self.decoder(h)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


## VAE Loss

Implement the reconstruction loss and KL divergence.

In [ ]:

def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    """
    Return total_loss, reconstruction_loss, kl_loss.

    Use binary cross entropy summed over pixels and averaged over batch.
    KL formula for diagonal Gaussian:

        KL(q(z|x) || N(0,I))
        = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))

    We average over the batch so that losses are comparable across batch sizes.
    """
    recon = nn.functional.binary_cross_entropy(
        x_hat, x, reduction="sum"
    ) / x.size(0)

    kl = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    ) / x.size(0)

    total = recon + beta * kl
    return total, recon, kl


In [ ]:
def train_vae(
    model,
    train_loader,
    epochs=5,
    lr=1e-3,
    beta=1.0,
    device=DEVICE,
):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"loss": [], "recon": [], "kl": []}

    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0.0
        total_recon = 0.0
        total_kl = 0.0
        n_seen = 0

        for xb, _ in train_loader:
            xb = xb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            x_hat, mu, logvar = model(xb)
            loss, recon, kl = vae_loss(x_hat, xb, mu, logvar, beta=beta)

            loss.backward()
            optimizer.step()

            batch_size = xb.size(0)
            n_seen += batch_size
            total_loss += loss.item() * batch_size
            total_recon += recon.item() * batch_size
            total_kl += kl.item() * batch_size

        avg_loss = total_loss / n_seen
        avg_recon = total_recon / n_seen
        avg_kl = total_kl / n_seen

        history["loss"].append(avg_loss)
        history["recon"].append(avg_recon)
        history["kl"].append(avg_kl)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"loss: {avg_loss:.2f} | "
            f"recon: {avg_recon:.2f} | "
            f"KL: {avg_kl:.2f}"
        )

    return history


## Task A: Train the VAE

Use a small latent dimension so the model runs quickly.

Recommended laptop settings:

- `latent_dim = 8`
- `epochs = 5`
- `batch_size = 128`
- `beta = 1.0`

If training is slow, use `epochs = 2`.


In [ ]:

LATENT_DIM = 8
BETA = 1.0
EPOCHS = 5

vae = ConvVAE(latent_dim=LATENT_DIM).to(DEVICE)
print("parameters:", count_parameters(vae))
print("Using device:", DEVICE)
history = train_vae(
    vae,
    train_loader,
    epochs=EPOCHS,
    lr=1e-3,
    beta=BETA,
    device=DEVICE,
)

plot_history(history)


## Visual Evaluation

Look at reconstructions, random samples from the prior, and latent statistics.

In [ ]:

@torch.no_grad()
def reconstruct_batch(model, loader, n=16, device=DEVICE):
    model.eval()
    xb, yb = next(iter(loader))
    xb = xb[:n].to(device)
    x_hat, mu, logvar = model(xb)
    return xb.cpu(), x_hat.cpu(), yb[:n], mu.cpu(), logvar.cpu()


@torch.no_grad()
def sample_from_prior(model, n=32, device=DEVICE):
    model.eval()
    z = torch.randn(n, model.latent_dim, device=device)
    samples = model.decode(z)
    return samples.cpu()


x_clean, x_recon, labels, mu, logvar = reconstruct_batch(vae, test_loader, n=16)

show_images(x_clean, title="Original test images", nrow=8)
show_images(x_recon, title="VAE reconstructions", nrow=8)

samples = sample_from_prior(vae, n=32)
show_images(samples, title="Samples generated from z ~ N(0, I)", nrow=8, figsize=(8, 4))

print("mu mean:", mu.mean(dim=0))
print("mu std:", mu.std(dim=0))
print("logvar mean:", logvar.mean(dim=0))


## Latent Interpolation

Encode two images, linearly interpolate between their latent means, and decode the path.

In [ ]:

@torch.no_grad()
def interpolate_two_images(model, x_a, x_b, steps=10, device=DEVICE):
    model.eval()

    x_pair = torch.stack([x_a, x_b]).to(device)
    mu, logvar = model.encode(x_pair)

    z_a = mu[0]
    z_b = mu[1]

    alphas = torch.linspace(0, 1, steps, device=device).unsqueeze(1)
    z = (1 - alphas) * z_a.unsqueeze(0) + alphas * z_b.unsqueeze(0)

    decoded = model.decode(z)
    return decoded.cpu()


# Pick two test images from different classes.
test_images, test_labels = next(iter(test_loader))

idx_a = 0
idx_b = 1

print("left:", class_names[test_labels[idx_a].item()])
print("right:", class_names[test_labels[idx_b].item()])

path = interpolate_two_images(
    vae,
    test_images[idx_a],
    test_images[idx_b],
    steps=12,
    device=DEVICE
)

show_images(path, title="Latent interpolation between two images", nrow=12, figsize=(10, 2))


## Optional: 2D VAE Latent Visualization

Train a small 2D VAE and plot the latent means. This is optional because it adds extra training time.

In [ ]:

# Optional visualization: train a 2D VAE briefly and plot its latent means.
# This is slower than just plotting reconstructions, but useful for visual intuition.

LATENT_DIM_2D = 2
EPOCHS_2D = 10

vae_2d = ConvVAE(latent_dim=LATENT_DIM_2D).to(DEVICE)

history_2d = train_vae(
    vae_2d,
    train_loader,
    epochs=EPOCHS_2D,
    lr=1e-3,
    beta=1.0,
    device=DEVICE,
)

@torch.no_grad()
def collect_latent_means(model, loader, max_batches=40, device=DEVICE):
    model.eval()
    mus = []
    ys = []

    for i, (xb, yb) in enumerate(loader):
        if i >= max_batches:
            break
        xb = xb.to(device)
        mu, logvar = model.encode(xb)
        mus.append(mu.cpu())
        ys.append(yb)

    return torch.cat(mus), torch.cat(ys)

Z, Y = collect_latent_means(vae_2d, test_loader, max_batches=40)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    Z[:, 0],
    Z[:, 1],
    c=Y,
    s=8,
    alpha=0.5,
    cmap="tab10"
)
cbar = plt.colorbar(scatter, ticks=list(range(10)))
cbar.ax.set_yticklabels(class_names)
plt.xlabel("latent mean coordinate 1")
plt.ylabel("latent mean coordinate 2")
plt.title("2D VAE latent means")
plt.grid(alpha=0.2)
plt.show()
